# 1. Instalando as Bibliotecas

In [56]:
!pip -q install \
langchain \
langchain-community \
langchain-groq \
langchain-text-splitters \
faiss-cpu \
sentence-transformers \
pypdf \
python-dotenv

# 2. Importando as bibliotecas

In [57]:
import os

from google.colab import userdata

# Documentos
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Divisão de texto
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Banco Vetorial
from langchain_community.vectorstores import FAISS

# Modelo Groq
from langchain_groq import ChatGroq

# Prompt
from langchain_core.prompts import ChatPromptTemplate

# Chains
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("✅ Bibliotecas carregadas.")

✅ Bibliotecas carregadas.


# 3. Clonar o repositório do GitHub

In [58]:
!rm -rf PortfolioAI

!git clone https://github.com/elissouza2023/PortfolioAI.git

BASE_PATH = "/content/PortfolioAI"

KNOWLEDGE_PATH = f"{BASE_PATH}/knowledge_base"

VECTOR_PATH = f"{BASE_PATH}/vector_store"

os.makedirs(VECTOR_PATH, exist_ok=True)

print("✅ Repositório clonado.")

Cloning into 'PortfolioAI'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 33 (delta 3), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 2.81 MiB | 6.28 MiB/s, done.
Resolving deltas: 100% (3/3), done.
✅ Repositório clonado.


# 4. Configurar API Key da Groq

In [59]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✅ API Key carregada.")

✅ API Key carregada.


# 5. Carregando documentos da pasta knowledge_base

In [60]:
loader = PyPDFDirectoryLoader(KNOWLEDGE_PATH)

documents = loader.load()

print(f"\n📄 Total de documentos: {len(documents)}")

for doc in documents:
    print(doc.metadata["source"])


📄 Total de documentos: 38
/content/PortfolioAI/knowledge_base/Trajetória Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Trajetória Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Trajetória Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Trajetória Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Projetos e Cases – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Projetos e Cases – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Projetos e Case

# 6. Dividindo os documentos em chunks

In [61]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=900,

    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ".",
        "!",
        "?",
        " "
    ]
)

texts = text_splitter.split_documents(documents)

print(f"✅ Chunks criados: {len(texts)}")

✅ Chunks criados: 83


# 7. Criando Embeddings

In [62]:
embeddings = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# 8. Banco Vetorial

In [63]:
vector_store = FAISS.from_documents(

    texts,

    embeddings

)

vector_store.save_local(VECTOR_PATH)

print("✅ Banco vetorial criado.")

✅ Banco vetorial criado.


# 9. Modelo Groq

In [64]:
MODEL_NAME = "llama-3.3-70b-versatile"

llm = ChatGroq(

    model_name=MODEL_NAME,

    temperature=0.2,

    max_tokens=1200

)

print("✅ Modelo carregado.")

✅ Modelo carregado.


## 10. Prompt do PortfolioAI

In [65]:
system_prompt = """
Você é o PortfolioAI.

Seu objetivo é responder perguntas sobre Elisângela de Souza.

REGRAS IMPORTANTES

• Utilize EXCLUSIVAMENTE as informações presentes no contexto.

• Nunca invente experiências.

• Nunca complete informações por conta própria.

• Caso não exista resposta no contexto, diga:

"Não encontrei essa informação na minha base de conhecimento.
Caso deseje mais detalhes, recomendo entrar em contato diretamente com Elisângela."

• Sempre escreva de forma profissional.

• Sempre responda em português.

• Quando possível organize a resposta em tópicos.

Contexto:

{context}
"""

prompt = ChatPromptTemplate.from_messages(

    [

        ("system", system_prompt),

        ("human", "{input}")

    ]

)

print("✅ Prompt criado.")

✅ Prompt criado.


# 11. Chain RAG

In [66]:
question_answer_chain = create_stuff_documents_chain(

    llm,

    prompt

)

retriever = vector_store.as_retriever(

    search_kwargs={

        "k":6

    }

)

rag_chain = create_retrieval_chain(

    retriever,

    question_answer_chain

)

print("✅ RAG criado.")

✅ RAG criado.


# 12. Função para perguntas

In [67]:
def perguntar(pergunta):

    resposta = rag_chain.invoke(

        {

            "input": pergunta

        }

    )

    print("="*80)

    print("PERGUNTA")

    print(pergunta)

    print()

    print("RESPOSTA")

    print(resposta["answer"])

    print()

    print("FONTES UTILIZADAS")

    fontes = set()

    for doc in resposta["context"]:

        fontes.add(

            os.path.basename(

                doc.metadata["source"]

            )

        )

    for fonte in sorted(fontes):

        print("•", fonte)

    print("="*80)

# 13. Testes

In [68]:
perguntar("Quem é Elisângela de Souza?")

PERGUNTA
Quem é Elisângela de Souza?

RESPOSTA
De acordo com as informações disponíveis, Elisângela de Souza é uma profissional com uma trajetória que combina experiência operacional e administrativa na indústria com uma transição ativa para a área de Tecnologia da Informação. Ela é bacharel em Administração, tem pós-graduação em Engenharia Metalúrgica e está concluindo a Tecnologia em Segurança da Informação.

Ela se descreve como uma profissional em constante evolução, com uma trajetória construída a partir da combinação entre experiência prática, aprendizado contínuo e interesse por tecnologia aplicada à resolução de problemas. Sua carreira iniciou-se em ambientes industriais e administrativos, onde desenvolveu competências relacionadas à operação, qualidade, gestão de processos, documentação técnica e melhoria contínua.

Atualmente, Elisângela está direcionando sua formação para a área de Tecnologia da Informação, consolidando conhecimentos em Desenvolvimento de Software, Inteligên

In [69]:
perguntar("Qual é o objetivo profissional dela?")

PERGUNTA
Qual é o objetivo profissional dela?

RESPOSTA
O objetivo profissional de Elisângela de Souza é atuar em projetos que permitam integrar Inteligência Artificial, desenvolvimento de software, dados e experiência do usuário para criar soluções inovadoras que gerem valor para pessoas e organizações. Ela busca contribuir com equipes colaborativas, compartilhando conhecimento, aprendendo continuamente e participando da construção de produtos digitais que aliem qualidade e sustentabilidade.

FONTES UTILIZADAS
• Competências Comportamentais – Elisângela de Souza.pdf
• Perfil Profissional – Elisângela de Souza.pdf
• Trajetória Profissional – Elisângela de Souza.pdf


In [70]:
perguntar("Quais projetos ela desenvolveu?")

PERGUNTA
Quais projetos ela desenvolveu?

RESPOSTA
De acordo com as informações disponíveis, Elisângela de Souza desenvolveu os seguintes projetos:

1. **PortfolioAI**: O próprio projeto que reúne os principais projetos desenvolvidos por ela.
2. **Kaida AI Risk Detector**: Uma ferramenta para identificar riscos de vazamento de dados em prompts de IA.
3. **Dashboard Mercado Siderúrgico Brasileiro**: Uma análise de dados do setor siderúrgico brasileiro utilizando Python e Streamlit.
4. **Flow State Shift**: Uma solução de UX para passagem de turno operacional em plantas industriais.

Esses projetos demonstram suas competências em áreas como Inteligência Artificial, Segurança da Informação, UX e desenvolvimento de soluções tecnológicas.

FONTES UTILIZADAS
• Competências Comportamentais – Elisângela de Souza.pdf
• Desenvolvimento Profissional Contínuo – Elisângela de Souza.pdf
• FAQ – Elisângela de Souza.pdf
• Projetos e Cases – Elisângela de Souza.pdf
• Trajetória Profissional – Elisângel

In [71]:
perguntar("Fale sobre o projeto PortfolioAI.")

PERGUNTA
Fale sobre o projeto PortfolioAI.

RESPOSTA
**PortfolioAI**

O PortfolioAI é o meu projeto principal. Ele é uma base de conhecimento que reúne os principais projetos desenvolvidos por mim ao longo de minha trajetória de formação e transição para a área de Tecnologia da Informação.

**Objetivo**

O objetivo do PortfolioAI é permitir que eu responda perguntas relacionadas à minha formação complementar, competências desenvolvidas e processo contínuo de aprendizagem. Além disso, ele serve como uma fonte oficial para fornecer contexto sobre como minha base educacional sustenta minha atuação profissional e minha evolução na área de tecnologia.

**Desenvolvimento**

Desenvolvi o PortfolioAI sozinha, aplicando na prática tudo que aprendi em Python, LangChain, embeddings e FAISS durante os cursos do Oracle Next Education. É um projeto 100% hands-on que demonstra minhas competências atuais.

**Tecnologias Utilizadas**

As tecnologias utilizadas no desenvolvimento do PortfolioAI incluem:

In [72]:
perguntar("Quais competências técnicas ela possui?")

PERGUNTA
Quais competências técnicas ela possui?

RESPOSTA
De acordo com o contexto fornecido, as competências técnicas de Elisângela de Souza incluem:

* Desenvolvimento de Software
* Conhecimentos específicos em:
 + Python
 + Programação Orientada a Objetos
 + Git
 + GitHub
 + SQL
 + Docker
 + APIs

Além disso, também são mencionadas as seguintes competências técnicas:

* Engenharia de Prompt
* IA Generativa
* UX Conversacional
* Automação

Essas competências técnicas são apresentadas como parte de sua base de conhecimento e são continuamente aprimoradas por meio de estudos, projetos práticos e desenvolvimento profissional.

FONTES UTILIZADAS
• Competências Comportamentais – Elisângela de Souza.pdf
• Competências Técnicas - Elisângela de Souza.pdf
• Projetos e Cases – Elisângela de Souza.pdf
• Trajetória Profissional – Elisângela de Souza.pdf


In [73]:
perguntar("Qual sua formação acadêmica?")

PERGUNTA
Qual sua formação acadêmica?

RESPOSTA
De acordo com as informações disponíveis, a formação acadêmica de Elisângela de Souza inclui:

* Graduação em Segurança da Informação
* Formação complementar, que inclui certificações, cursos de aperfeiçoamento e projetos práticos, apresentados no documento 04 – Formação Complementar desta Base de Conhecimento.

Essa formação acadêmica foi construída ao longo da carreira, acompanhando a evolução profissional e refletindo a busca constante por conhecimento, atualização e desenvolvimento técnico. Além disso, Elisângela de Souza também valoriza o aprendizado contínuo e a atualização constante para acompanhar a evolução das tecnologias e das necessidades do mercado.

FONTES UTILIZADAS
• Competências Comportamentais – Elisângela de Souza.pdf
• Competências Técnicas - Elisângela de Souza.pdf
• Formação Acadêmica - Elisângela de Souza.pdf
